<a href="https://colab.research.google.com/github/gdhameja1/ML/blob/main/RiskRadar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
!pip install -q anthropic networkx pandas matplotlib

2.11.0+cu128 True Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.2 MB/s eta 0:00:00


In [ ]:
import json
import re
from datetime import datetime, timedelta
from getpass import getpass

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import anthropic

API_KEY = getpass("Enter your Anthropic API key: ")
client = anthropic.Anthropic(api_key=API_KEY)

# Current Claude models (Aug 2026). Sonnet is the balanced default for this kind of
# extraction + reasoning task; swap to claude-haiku-4-5-20251001 for lower cost/latency
# at higher volume, or claude-opus-4-8 if you want deeper reasoning on ambiguous updates.
MODEL = "claude-sonnet-5"


In [ ]:
now = datetime.utcnow()

SAMPLE_UPDATES = [
    {
        "source": "jira", "entity": "PROJ-104", "actor": "sara.k",
        "timestamp": (now - timedelta(days=6)).isoformat(),
        "event_type": "comment",
        "raw_text": "Still waiting on the payments team to expose the new webhook endpoint. "
                     "Can't finish integration testing until that's live. This is the third day blocked."
    },
    {
        "source": "slack", "entity": "#team-checkout", "actor": "raj.p",
        "timestamp": (now - timedelta(days=1)).isoformat(),
        "event_type": "message",
        "raw_text": "Heads up, the vendor SDK we depend on for fraud checks just deprecated the "
                     "method we use. We'll need to migrate before the Oct release or checkout breaks."
    },
    {
        "source": "github", "entity": "PR #482", "actor": "mina.z",
        "timestamp": (now - timedelta(hours=10)).isoformat(),
        "event_type": "pr_comment",
        "raw_text": "This PR has been open for 9 days with 3 failed CI runs. Merge conflicts keep "
                     "reappearing because it depends on the auth-refactor branch that hasn't landed."
    },
    {
        "source": "jira", "entity": "PROJ-110", "actor": "sara.k",
        "timestamp": (now - timedelta(days=2)).isoformat(),
        "event_type": "status_change",
        "raw_text": "Moved back to In Progress from Done. QA found the reporting numbers don't "
                     "match finance's expectations, scope may be bigger than estimated."
    },
    {
        "source": "confluence", "entity": "RFC: Notification Service v2", "actor": "devon.l",
        "timestamp": (now - timedelta(days=3)).isoformat(),
        "event_type": "doc_edit",
        "raw_text": "Updated the RFC — we now need sign-off from the security team before this can "
                     "proceed, which wasn't in the original plan. No response from security yet."
    },
    {
        "source": "slack", "entity": "#eng-standup", "actor": "raj.p",
        "timestamp": (now - timedelta(hours=20)).isoformat(),
        "event_type": "message",
        "raw_text": "Quick update: fixed the flaky test suite, everything green now, on track for Friday."
    },
    {
        "source": "jira", "entity": "PROJ-104", "actor": "sara.k",
        "timestamp": (now - timedelta(days=4)).isoformat(),
        "event_type": "comment",
        "raw_text": "Pinged payments team again about the webhook. No response in 2 days."
    },
]

df_raw = pd.DataFrame(SAMPLE_UPDATES)
df_raw

/tmp/ipykernel_1265/70256004.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


,source,entity,actor,timestamp,event_type,raw_text
0,jira,PROJ-104,sara.k,2026-08-17T15:19:16.741218,comment,Still waiting on the payments team to expose t...
1,slack,#team-checkout,raj.p,2026-08-22T15:19:16.741218,message,"Heads up, the vendor SDK we depend on for frau..."
2,github,PR #482,mina.z,2026-08-23T05:19:16.741218,pr_comment,This PR has been open for 9 days with 3 failed...
3,jira,PROJ-110,sara.k,2026-08-21T15:19:16.741218,status_change,Moved back to In Progress from Done. QA found ...
4,confluence,RFC: Notification Service v2,devon.l,2026-08-20T15:19:16.741218,doc_edit,Updated the RFC — we now need sign-off from th...
5,slack,#eng-standup,raj.p,2026-08-22T19:19:16.741218,message,"Quick update: fixed the flaky test suite, ever..."
6,jira,PROJ-104,sara.k,2026-08-19T15:19:16.741218,comment,Pinged payments team again about the webhook. ...


In [ ]:
EXTRACTION_SYSTEM_PROMPT = '''You are a project risk analyst. Given one project update, extract
structured signals. Respond with ONLY a JSON object, no other text, matching this schema:

{
  "has_signal": boolean,           // true if this update contains a risk, dependency, or blocker
  "signal_types": [string],        // any of: "risk", "dependency", "blocker", "none"
  "summary": string,               // one-sentence plain-language summary of the concern
  "blocked_by": string or null,    // what/who this is waiting on, if named
  "severity_hint": "low"|"medium"|"high",  // your own judgment of urgency/impact
  "sentiment": "neutral"|"concerned"|"escalating"
}

Rules:
- If the update is just routine status ("on track", "done", "shipped"), set has_signal to false.
- severity_hint "high" means: could slip a deadline, blocks other teams, or has been recurring.
- Be conservative — don't invent blockers that aren't stated or clearly implied.'''

def extract_signal(update):
    prompt = f"""Source: {update['source']}
Entity: {update['entity']}
Author: {update['actor']}
Update: {update['raw_text']}"""

    response = client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.content[0].text.strip()
    text = re.sub(r"^```json\s*|\s*```$", "", text.strip())
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"has_signal": False, "signal_types": ["none"], "summary": "(parse error)",
                "blocked_by": None, "severity_hint": "low", "sentiment": "neutral"}


In [ ]:
extracted = []
for _, row in df_raw.iterrows():
    sig = extract_signal(row)
    extracted.append({**row.to_dict(), **sig})

df = pd.DataFrame(extracted)
flagged = df[df["has_signal"] == True].copy()
flagged[["entity", "source", "signal_types", "summary", "blocked_by", "severity_hint", "sentiment"]]

NameError: name 'client' is not defined

In [ ]:
G = nx.DiGraph()

for _, row in flagged.iterrows():
    G.add_node(row["entity"], source=row["source"])
    if row["blocked_by"]:
        G.add_node(row["blocked_by"])
        G.add_edge(row["entity"], row["blocked_by"], relation="blocked_by")

# downstream_count = how many flagged items are blocked, directly or indirectly, on this node
def downstream_count(node):
    if node not in G:
        return 0
    return len(nx.ancestors(G, node))

plt.figure(figsize=(8, 5))
pos = nx.spring_layout(G, seed=7)
nx.draw(G, pos, with_labels=True, node_color="#f4a259", node_size=1800,
        font_size=8, arrows=True, edge_color="#555")
plt.title("Dependency graph: item -> blocked_by")
plt.show()


NameError: name 'flagged' is not defined

In [ ]:
SEVERITY_WEIGHTS = {"low": 1, "medium": 2, "high": 3}

recurrence = flagged.groupby("entity").size().to_dict()

def score_row(row):
    base = SEVERITY_WEIGHTS.get(row["severity_hint"], 1)
    recur_bonus = min(recurrence.get(row["entity"], 1) - 1, 3)  # cap bonus at 3
    downstream_bonus = downstream_count(row["entity"])
    age_days = (now - pd.to_datetime(row["timestamp"])).days
    age_bonus = min(age_days // 2, 3)  # older unresolved items creep up the list
    return base * 3 + recur_bonus + downstream_bonus + age_bonus

flagged["severity_score"] = flagged.apply(score_row, axis=1)
ranked = flagged.sort_values("severity_score", ascending=False)
ranked[["entity", "signal_types", "summary", "severity_hint", "severity_score"]]

NameError: name 'flagged' is not defined

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(ranked["entity"] + " (" + ranked["source"] + ")", ranked["severity_score"], color="#e07a5f")
plt.xlabel("Severity score")
plt.title("Flagged items ranked by severity")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

NameError: name 'ranked' is not defined

<Figure size 800x400 with 0 Axes>

In [ ]:
def generate_digest(ranked_df, top_n=5):
    items_text = "\n".join(
        f"- [{r.severity_score:.0f}] {r.entity} ({r.source}): {r.summary} "
        f"(blocked by: {r.blocked_by or 'n/a'}, sentiment: {r.sentiment})"
        for r in ranked_df.head(top_n).itertuples()
    )
    prompt = f"""Here are this period's top flagged project items, most severe first:

{items_text}

Write a short daily risk digest for an engineering manager: 3-5 sentences, plain language,
group related items if they share a root cause, and end with one clear recommended action.
No headers, no bullet points, just a tight narrative paragraph."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text.strip()

digest = generate_digest(ranked)
print(digest)


NameError: name 'ranked' is not defined

In [ ]:
ALERT_THRESHOLD = 9  # tune based on your own score distribution

def send_alert(row):
    # Placeholder — replace with a real Slack webhook POST or email send.
    print(f"ALERT: {row['entity']} (score {row['severity_score']:.0f}) — {row['summary']}")

high_priority = ranked[ranked["severity_score"] >= ALERT_THRESHOLD]
for _, row in high_priority.iterrows():
    send_alert(row)

if high_priority.empty:
    print("No items above the real-time alert threshold this cycle.")


NameError: name 'ranked' is not defined

In [ ]:
FEEDBACK_LOG_PATH = "feedback_log.csv"

def record_feedback(entity, verdict, notes=""):
    """verdict: 'valid' | 'noise' | 'handled'"""
    entry = pd.DataFrame([{
        "entity": entity, "verdict": verdict, "notes": notes,
        "logged_at": datetime.utcnow().isoformat()
    }])
    try:
        existing = pd.read_csv(FEEDBACK_LOG_PATH)
        entry = pd.concat([existing, entry], ignore_index=True)
    except FileNotFoundError:
        pass
    entry.to_csv(FEEDBACK_LOG_PATH, index=False)
    return entry

# Example usage:
# record_feedback("PROJ-104", "valid", "escalated to payments lead")
